In [1]:
import os
import sys

import h5py
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch

sys.path.insert(0, os.path.abspath("../"))
from optbnn.bnn.likelihoods import LikCE
from optbnn.bnn.nets.mlp import MLP
from optbnn.bnn.priors import OptimGaussianPrior
from optbnn.sgmcmc_bayes_net.pref_net import PrefNet
from optbnn.utils import util

In [2]:
# with h5py.File("../data/bb/state_data_1/tl-fullauto.hdf5") as f:
#     print(f.keys())
#     ai_1 = f["states"][:, 0, 23] == 1.0
#     ai_2 = f["states"][:, 0, 23] == 2.0
#     ai_3 = f["states"][:, 0, 23] == 3.0
#     ai_4 = f["states"][:, 0, 23] == 4.0

#     states_1 = f["states"][ai_1, ...]
#     actions_1 = f["actions"][ai_1, ...]
#     timesteps_1 = f["timesteps"][ai_1, ...]
#     attn_mask_1 = f["attn_mask"][ai_1, ...]

#     states_2 = f["states"][ai_2, ...]
#     actions_2 = f["actions"][ai_2, ...]
#     timesteps_2 = f["timesteps"][ai_2, ...]
#     attn_mask_2 = f["attn_mask"][ai_2, ...]

#     states_3 = f["states"][ai_3, ...]
#     actions_3 = f["actions"][ai_3, ...]
#     timesteps_3 = f["timesteps"][ai_3, ...]
#     attn_mask_3 = f["attn_mask"][ai_3, ...]

#     states_4 = f["states"][ai_4, ...]
#     actions_4 = f["actions"][ai_4, ...]
#     timesteps_4 = f["timesteps"][ai_4, ...]
#     attn_mask_4 = f["attn_mask"][ai_4, ...]

In [3]:
t0012_prior_path = "./../exp/reward_learning/bb_tuning_star/br-BB_t0012-256-3"
t0012_ckpt_path = os.path.join(t0012_prior_path, "ckpts", "it-{}.ckpt".format(300))
t0012_saved_dir = os.path.join(
    "./../exp/reward_learning/bb_optim/br-BB_t0012-a76832c6", "sampling_optim"
)
width = 256
depth = 3
transfer_fn = "relu"
t0012_prior = OptimGaussianPrior(t0012_ckpt_path)
t0012_net = MLP(28, 1, [width] * depth, transfer_fn)
t0012_likelihood = LikCE()
t0012_bayes_net = PrefNet(
    t0012_net, t0012_likelihood, t0012_prior, t0012_saved_dir, n_gpu=0
)
t0012_bayes_net.sampled_weights = t0012_bayes_net._load_sampled_weights(
    "./../exp/reward_learning/bb_optim/br-BB_t0012-a76832c6/sampling_optim/sampled_weights/sampled_weights_0000003"
)

t0048_prior_path = "./../exp/reward_learning/bb_tuning_star/br-BB_t0048-256-3"
t0048_ckpt_path = os.path.join(t0048_prior_path, "ckpts", "it-{}.ckpt".format(300))
t0048_saved_dir = os.path.join(
    "./../exp/reward_learning/bb_optim/br-BB_t0048-384f3849", "sampling_optim"
)
width = 256
depth = 3
transfer_fn = "relu"
t0048_prior = OptimGaussianPrior(t0048_ckpt_path)
t0048_net = MLP(28, 1, [width] * depth, transfer_fn)
t0048_likelihood = LikCE()
t0048_bayes_net = PrefNet(
    t0048_net, t0048_likelihood, t0048_prior, t0048_saved_dir, n_gpu=0
)
t0048_bayes_net.sampled_weights = t0048_bayes_net._load_sampled_weights(
    "./../exp/reward_learning/bb_optim/br-BB_t0048-384f3849/sampling_optim/sampled_weights/sampled_weights_0000003"
)

<class 'numpy.ndarray'>


In [4]:
def compute_posterior_returns(rewards, gamma=1.0):
    returns = np.zeros_like(rewards, dtype=np.float32)
    running_return = np.zeros(rewards.shape[1])
    for t in reversed(range(rewards.shape[0])):
        running_return = rewards[t] + gamma * running_return
        returns[t] = running_return
    return returns

In [5]:
with h5py.File("../data/bb/state_data_1/a1.hdf5", "a") as f:
    returns_t0012 = []
    max_size_t0012 = -np.inf

    returns_t0048 = []
    max_size_t0048 = -np.inf
    for i in range(f["states"].shape[0]):
        am = f["attn_mask"][i]
        sts = f["states"][i][: int(am.sum()), ...]
        acts = f["actions"][i][: int(am.sum()), ...]
        _, _, t0012_preds = t0012_bayes_net.predict(
            np.concatenate([sts, acts], axis=-1), True
        )
        t0012_preds = np.squeeze(t0012_preds, axis=-1).T
        returns_t0012.append(compute_posterior_returns(t0012_preds))
        if max_size_t0012 < t0012_preds.shape[0]:
            max_size_t0012 = t0012_preds.shape[0]

        _, _, t0048_preds = t0048_bayes_net.predict(
            np.concatenate([sts, acts], axis=-1), True
        )
        t0048_preds = np.squeeze(t0048_preds, axis=-1).T
        returns_t0048.append(compute_posterior_returns(t0048_preds))
        if max_size_t0048 < t0048_preds.shape[0]:
            max_size_t0048 = t0048_preds.shape[0]
    returns_t0012 = np.stack(
        [
            np.pad(
                r,
                ((0, max_size_t0012 - r.shape[0]), (0, 0)),
                mode="constant",
                constant_values=np.nan,
            )
            for r in returns_t0012
        ]
    )

    returns_t0048 = np.stack(
        [
            np.pad(
                r,
                ((0, max_size_t0048 - r.shape[0]), (0, 0)),
                mode="constant",
                constant_values=np.nan,
            )
            for r in returns_t0048
        ]
    )
    r_dist_t0012 = returns_t0012.ravel()
    r_dist_1_t0012 = r_dist_t0012[~np.isnan(r_dist_t0012)]

    Q1 = np.percentile(r_dist_1_t0012, 25)
    Q3 = np.percentile(r_dist_1_t0012, 75)
    IQR = Q3 - Q1

    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    # r_dist_1_t0012 = np.clip(r_dist_1_t0012, lower_bound, upper_bound)

    r_dist_1_t0012 = r_dist_1_t0012[
        (r_dist_1_t0012 >= lower_bound) & (r_dist_1_t0012 <= upper_bound)
    ]

    r_dist_t0048 = returns_t0048.ravel()
    r_dist_1_t0048 = r_dist_t0048[~np.isnan(r_dist_t0048)]

    Q1 = np.percentile(r_dist_1_t0048, 25)
    Q3 = np.percentile(r_dist_1_t0048, 75)
    IQR = Q3 - Q1

    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    # r_dist_1_t0048 = np.clip(r_dist_1_t0048, lower_bound, upper_bound)

    r_dist_1_t0048 = r_dist_1_t0048[
        (r_dist_1_t0048 >= lower_bound) & (r_dist_1_t0048 <= upper_bound)
    ]

    # f.create_dataset("states", data=states_1)
    # f.create_dataset("actions", data=actions_1)
    # f.create_dataset("timesteps", data=timesteps_1)
    # f.create_dataset("attn_mask", data=attn_mask_1)

KeyError: "Unable to synchronously open object (object 'states' doesn't exist)"

In [ ]:
with h5py.File("../data/bb/state_data_1/a2.hdf5", "a") as f:
    returns_t0012 = []
    max_size_t0012 = -np.inf

    returns_t0048 = []
    max_size_t0048 = -np.inf
    for i in range(f["states"].shape[0]):
        am = f["attn_mask"][i]
        sts = f["states"][i][: int(am.sum()), ...]
        acts = f["actions"][i][: int(am.sum()), ...]
        _, _, t0012_preds = t0012_bayes_net.predict(
            np.concatenate([sts, acts], axis=-1), True
        )
        t0012_preds = np.squeeze(t0012_preds, axis=-1).T
        returns_t0012.append(compute_posterior_returns(t0012_preds))
        if max_size_t0012 < t0012_preds.shape[0]:
            max_size_t0012 = t0012_preds.shape[0]

        _, _, t0048_preds = t0048_bayes_net.predict(
            np.concatenate([sts, acts], axis=-1), True
        )
        t0048_preds = np.squeeze(t0048_preds, axis=-1).T
        returns_t0048.append(compute_posterior_returns(t0048_preds))
        if max_size_t0048 < t0048_preds.shape[0]:
            max_size_t0048 = t0048_preds.shape[0]
    returns_t0012 = np.stack(
        [
            np.pad(
                r,
                ((0, max_size_t0012 - r.shape[0]), (0, 0)),
                mode="constant",
                constant_values=np.nan,
            )
            for r in returns_t0012
        ]
    )

    returns_t0048 = np.stack(
        [
            np.pad(
                r,
                ((0, max_size_t0048 - r.shape[0]), (0, 0)),
                mode="constant",
                constant_values=np.nan,
            )
            for r in returns_t0048
        ]
    )
    r_dist_t0012 = returns_t0012.ravel()
    r_dist_2_t0012 = r_dist_t0012[~np.isnan(r_dist_t0012)]

    Q1 = np.percentile(r_dist_2_t0012, 25)
    Q3 = np.percentile(r_dist_2_t0012, 75)
    IQR = Q3 - Q1

    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    # r_dist_2_t0012 = np.clip(r_dist_2_t0012, lower_bound, upper_bound)

    r_dist_2_t0012 = r_dist_2_t0012[
        (r_dist_2_t0012 >= lower_bound) & (r_dist_2_t0012 <= upper_bound)
    ]

    r_dist_t0048 = returns_t0048.ravel()
    r_dist_2_t0048 = r_dist_t0048[~np.isnan(r_dist_t0048)]

    Q1 = np.percentile(r_dist_2_t0048, 25)
    Q3 = np.percentile(r_dist_2_t0048, 75)
    IQR = Q3 - Q1

    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    # r_dist_2_t0048 = np.clip(r_dist_2_t0048, lower_bound, upper_bound)

    r_dist_2_t0048 = r_dist_2_t0048[
        (r_dist_2_t0048 >= lower_bound) & (r_dist_2_t0048 <= upper_bound)
    ]

    # f.create_dataset("states", data=states_2)
    # f.create_dataset("actions", data=actions_2)
    # f.create_dataset("timesteps", data=timesteps_2)
    # f.create_dataset("attn_mask", data=attn_mask_2)

In [ ]:
with h5py.File("../data/bb/state_data_1/a3.hdf5", "a") as f:
    returns_t0012 = []
    max_size_t0012 = -np.inf

    returns_t0048 = []
    max_size_t0048 = -np.inf
    for i in range(f["states"].shape[0]):
        am = f["attn_mask"][i]
        sts = f["states"][i][: int(am.sum()), ...]
        acts = f["actions"][i][: int(am.sum()), ...]
        _, _, t0012_preds = t0012_bayes_net.predict(
            np.concatenate([sts, acts], axis=-1), True
        )
        t0012_preds = np.squeeze(t0012_preds, axis=-1).T
        returns_t0012.append(compute_posterior_returns(t0012_preds))
        if max_size_t0012 < t0012_preds.shape[0]:
            max_size_t0012 = t0012_preds.shape[0]

        _, _, t0048_preds = t0048_bayes_net.predict(
            np.concatenate([sts, acts], axis=-1), True
        )
        t0048_preds = np.squeeze(t0048_preds, axis=-1).T
        returns_t0048.append(compute_posterior_returns(t0048_preds))
        if max_size_t0048 < t0048_preds.shape[0]:
            max_size_t0048 = t0048_preds.shape[0]
    returns_t0012 = np.stack(
        [
            np.pad(
                r,
                ((0, max_size_t0012 - r.shape[0]), (0, 0)),
                mode="constant",
                constant_values=np.nan,
            )
            for r in returns_t0012
        ]
    )

    returns_t0048 = np.stack(
        [
            np.pad(
                r,
                ((0, max_size_t0048 - r.shape[0]), (0, 0)),
                mode="constant",
                constant_values=np.nan,
            )
            for r in returns_t0048
        ]
    )
    r_dist_t0012 = returns_t0012.ravel()
    r_dist_3_t0012 = r_dist_t0012[~np.isnan(r_dist_t0012)]

    Q1 = np.percentile(r_dist_3_t0012, 25)
    Q3 = np.percentile(r_dist_3_t0012, 75)
    IQR = Q3 - Q1

    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    # r_dist_3_t0012 = np.clip(r_dist_3_t0012, lower_bound, upper_bound)

    r_dist_3_t0012 = r_dist_3_t0012[
        (r_dist_3_t0012 >= lower_bound) & (r_dist_3_t0012 <= upper_bound)
    ]

    r_dist_t0048 = returns_t0048.ravel()
    r_dist_3_t0048 = r_dist_t0048[~np.isnan(r_dist_t0048)]

    Q1 = np.percentile(r_dist_3_t0048, 25)
    Q3 = np.percentile(r_dist_3_t0048, 75)
    IQR = Q3 - Q1

    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    # r_dist_3_t0048 = np.clip(r_dist_3_t0048, lower_bound, upper_bound)

    r_dist_3_t0048 = r_dist_3_t0048[
        (r_dist_3_t0048 >= lower_bound) & (r_dist_3_t0048 <= upper_bound)
    ]

    # f.create_dataset("states", data=states_3)
    # f.create_dataset("actions", data=actions_3)
    # f.create_dataset("timesteps", data=timesteps_3)
    # f.create_dataset("attn_mask", data=attn_mask_3)

In [ ]:
with h5py.File("../data/bb/state_data_1/a4.hdf5", "a") as f:
    returns_t0012 = []
    max_size_t0012 = -np.inf

    returns_t0048 = []
    max_size_t0048 = -np.inf
    for i in range(f["states"].shape[0]):
        am = f["attn_mask"][i]
        sts = f["states"][i][: int(am.sum()), ...]
        acts = f["actions"][i][: int(am.sum()), ...]
        _, _, t0012_preds = t0012_bayes_net.predict(
            np.concatenate([sts, acts], axis=-1), True
        )
        t0012_preds = np.squeeze(t0012_preds, axis=-1).T
        returns_t0012.append(compute_posterior_returns(t0012_preds))
        if max_size_t0012 < t0012_preds.shape[0]:
            max_size_t0012 = t0012_preds.shape[0]

        _, _, t0048_preds = t0048_bayes_net.predict(
            np.concatenate([sts, acts], axis=-1), True
        )
        t0048_preds = np.squeeze(t0048_preds, axis=-1).T
        returns_t0048.append(compute_posterior_returns(t0048_preds))
        if max_size_t0048 < t0048_preds.shape[0]:
            max_size_t0048 = t0048_preds.shape[0]
    returns_t0012 = np.stack(
        [
            np.pad(
                r,
                ((0, max_size_t0012 - r.shape[0]), (0, 0)),
                mode="constant",
                constant_values=np.nan,
            )
            for r in returns_t0012
        ]
    )

    returns_t0048 = np.stack(
        [
            np.pad(
                r,
                ((0, max_size_t0048 - r.shape[0]), (0, 0)),
                mode="constant",
                constant_values=np.nan,
            )
            for r in returns_t0048
        ]
    )
    r_dist_t0012 = returns_t0012.ravel()
    r_dist_4_t0012 = r_dist_t0012[~np.isnan(r_dist_t0012)]

    Q1 = np.percentile(r_dist_4_t0012, 25)
    Q3 = np.percentile(r_dist_4_t0012, 75)
    IQR = Q3 - Q1

    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    # r_dist_4_t0012 = np.clip(r_dist_4_t0012, lower_bound, upper_bound)

    r_dist_4_t0012 = r_dist_4_t0012[
        (r_dist_4_t0012 >= lower_bound) & (r_dist_4_t0012 <= upper_bound)
    ]

    r_dist_t0048 = returns_t0048.ravel()
    r_dist_4_t0048 = r_dist_t0048[~np.isnan(r_dist_t0048)]

    Q1 = np.percentile(r_dist_4_t0048, 25)
    Q3 = np.percentile(r_dist_4_t0048, 75)
    IQR = Q3 - Q1

    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    # r_dist_4_t0048 = np.clip(r_dist_4_t0048, lower_bound, upper_bound)

    r_dist_4_t0048 = r_dist_4_t0048[
        (r_dist_4_t0048 >= lower_bound) & (r_dist_4_t0048 <= upper_bound)
    ]

    # f.create_dataset("states", data=states_4)
    # f.create_dataset("actions", data=actions_4)
    # f.create_dataset("timesteps", data=timesteps_4)
    # f.create_dataset("attn_mask", data=attn_mask_4)

In [ ]:
data = {
    "AI_1": [r_dist_1_t0012, r_dist_1_t0048],
    "AI_2": [r_dist_2_t0012, r_dist_2_t0048],
    "AI_3": [r_dist_3_t0012, r_dist_3_t0048],
    "AI_4": [r_dist_4_t0012, r_dist_4_t0048],
}

rows = []
for group_name, arrays in data.items():
    for i, arr in enumerate(arrays):
        if i == 0:
            part = "t0012"
        else:
            part = "t0048"
        for value in arr:
            rows.append({"Return": value, "Participant": f"{part}", "AI": group_name})

df = pd.DataFrame(rows)
plt.figure(figsize=(10, 6))
sns.violinplot(data=df, x="AI", y="Return", hue="Participant", cut=0)
plt.title("Return Distributions")
plt.show()

In [ ]:
var_05 = df.groupby(["Participant", "AI"])["Return"].quantile(0.05)

In [ ]:
var_05.reset_index().pivot(columns="Participant", index="AI", values="Return")

In [ ]:
thresholds = df.groupby(["Participant", "AI"])["Return"].transform("quantile", 0.05)
cvar_05 = df[df["Return"] <= thresholds].groupby(["Participant", "AI"])["Return"].mean()
cvar_05.reset_index().pivot(columns="Participant", index="AI", values="Return")